In [12]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
from statsmodels.tsa.stattools import grangercausalitytests
lag_order = 1 # since we aggregated the data in to 9 bins we only need 1 lag
maxlag = (
    lag_order,  # becuase we got this value before. We are not suppose to add 1 to it
)
test = "ssr_chi2test"
import scanpy as sc
from joblib import Parallel, delayed

In [2]:
def grangers_causation_matrix(
    data, in_variables, out_variables, test="ssr_chi2test", n_jobs=1, warn=False
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """

    def get_pval(dd):
        if warn:
            test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=True)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=FutureWarning)
                test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=False)
                # according to the documentation https://www.statsmodels.org/dev/generated/statsmodels.tsa.stattools.grangercausalitytests.html,
                # the dd has 2 columns, second causes the first.

        p_values = [test_result[i][0][test][1] for i in maxlag] # test_result[i][1] is the unrestricted model, test_result[i][1][0] is the restricted model
        coefs = [test_result[i][1][1].params[1] for i in maxlag] # x1, x2, const

        arg_min_p_value = np.argmin(p_values)
        min_p_value = p_values[arg_min_p_value]
        min_coef = coefs[arg_min_p_value]
        return (min_p_value, min_coef)

    out = Parallel(n_jobs=n_jobs)(
        delayed(get_pval)(data[[c, r]]) # this means r causes c, so r is be in and c is out
        for c in tqdm(out_variables, desc="Processing columns")  # Outer loop progress bar
        for r in in_variables  # Inner loop without progress bar
    )
    out_p = [p for (p,c) in out]
    out_c = [c for (p,c) in out]
    df_p = pd.DataFrame(
        np.array(out_p).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T # used the correct reshaping, and then transposed the matrix so the x and y are semantically correct (x causes y).
    df_c = pd.DataFrame(
        np.array(out_c).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T
    df_p.index = [var + "_x" for var in in_variables]
    df_p.columns = [var + "_y" for var in out_variables]
    df_c.index = [var + "_x" for var in in_variables]
    df_c.columns = [var + "_y" for var in out_variables]
    return df_p, df_c

def do_granger(trajs, in_genes, out_genes, n_jobs=1, warn=False):
    # in causes out
    trajs = trajs.T[::10]
    trajs = trajs - trajs.shift(1)
    trajs = trajs.dropna()
    out_traj_p, out_traj_c = grangers_causation_matrix(
        trajs, in_variables=in_genes, out_variables=out_genes, n_jobs=n_jobs, warn=warn
    )
    return out_traj_p, out_traj_c

# All Genes

In [3]:
trajectories = np.load('../../results/scRNAseq/trajectories_gene_space.npy', allow_pickle=True)
adata = sc.read('../../data/processed/adata_mioflow.h5ad')
genes = adata.var_names.to_list()
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
hv_mask = adata.var['highly_variable']
genes_hvg = hv_mask[hv_mask].index.tolist()
col_genes = np.array(genes_hvg)

In [4]:
db_extract = pd.read_csv('DatabaseExtract_v_1.01.csv')
db_ensembl_ids = set(db_extract['Ensembl ID'].astype(str))
genes_ensembl_ids = set([g.split('(')[-1].replace(')', '').strip() for g in genes_hvg])
in_genes_ensembl = genes_ensembl_ids & db_ensembl_ids
in_genes = [g for g in genes_hvg if g.split('(')[-1].replace(')', '').strip() in in_genes_ensembl]

In [5]:
trajectories_hvg = trajectories[:, :, hv_mask.values]
avg_traj = trajectories_hvg.mean(axis=1)
trajectories_df = pd.DataFrame(avg_traj, columns=col_genes)
out_traj_p, out_traj_c = do_granger(trajectories_df.T, in_genes, genes_hvg, n_jobs=1, warn=False)

Processing columns: 100%|██████████| 2000/2000 [05:52<00:00,  5.67it/s]


In [6]:
out_traj_p.shape, out_traj_c.shape

((303, 2000), (303, 2000))

In [8]:
out_traj_p

,A2M (ENSG00000175899)_y,ABCD4 (ENSG00000119688)_y,ABHD14B (ENSG00000114779)_y,AC003090.1 (ENSG00000223561)_y,AC007292.3 (ENSG00000269318)_y,AC007325.4 (ENSG00000278817)_y,AC009501.4 (ENSG00000231609)_y,AC012358.8 (ENSG00000240401)_y,AC092835.2 (ENSG00000233757)_y,AC093642.3 (ENSG00000237940)_y,...,ZNF503-AS1 (ENSG00000226051)_y,ZNF589 (ENSG00000164048)_y,ZNF592 (ENSG00000166716)_y,ZNF667-AS1 (ENSG00000166770)_y,ZNF69 (ENSG00000198429)_y,ZNF804A (ENSG00000170396)_y,ZNF93 (ENSG00000184635)_y,ZWINT (ENSG00000122952)_y,ZYG11B (ENSG00000162378)_y,ZZEF1 (ENSG00000074755)_y
AC092835.2 (ENSG00000233757)_x,1.895213e-01,7.751318e-06,9.704726e-02,7.914752e-01,7.024807e-03,8.237946e-01,2.520824e-03,2.942714e-05,1.000000,0.004497,...,5.092717e-01,1.715446e-02,6.943806e-01,2.449118e-01,5.514670e-01,0.028719,2.160997e-01,4.550435e-02,1.754379e-03,2.445728e-01
AES (ENSG00000104964)_x,6.960318e-49,4.976189e-01,6.392397e-01,9.309913e-81,1.828696e-03,0.000000e+00,1.451850e-54,1.268095e-01,0.000002,0.112031,...,1.097538e-02,2.051757e-110,4.355855e-04,1.783156e-22,1.532218e-141,0.267952,7.907748e-48,2.708363e-01,4.889537e-02,2.566485e-108
AGT (ENSG00000135744)_x,4.834397e-01,1.654553e-17,2.356723e-01,2.370475e-01,2.806819e-22,6.883337e-02,2.413036e-03,6.410123e-48,0.435359,0.343688,...,1.266680e-01,5.115687e-01,1.689323e-03,7.294144e-01,6.028098e-02,0.844064,6.931218e-01,6.463220e-01,5.896330e-02,9.413158e-03
ALX1 (ENSG00000180318)_x,2.663933e-01,6.017572e-02,2.827683e-09,3.123468e-01,8.762213e-03,8.503231e-01,1.880576e-06,1.919893e-09,0.295409,0.046238,...,1.512631e-06,8.797991e-02,2.361777e-07,7.906865e-01,5.852257e-02,0.844233,5.716367e-01,3.164693e-03,4.538017e-01,8.776907e-03
ALX3 (ENSG00000156150)_x,5.170876e-01,8.815438e-07,7.290854e-02,4.879350e-01,6.961613e-05,3.728508e-01,1.341949e-01,1.981770e-02,0.272493,0.096037,...,7.033250e-04,5.804488e-01,6.074036e-08,9.492289e-01,8.957404e-01,0.847764,8.785246e-01,4.339869e-06,4.503207e-01,7.347144e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF589 (ENSG00000164048)_x,5.153990e-05,6.488268e-10,5.867807e-02,1.973446e-14,4.575715e-21,2.883799e-09,5.613511e-86,5.809035e-09,0.001908,0.595575,...,5.528863e-01,1.000000e+00,3.595454e-08,1.268376e-09,2.211613e-11,0.995321,2.594232e-15,5.123879e-01,4.287654e-01,8.311301e-17
ZNF592 (ENSG00000166716)_x,3.515346e-01,2.446509e-03,2.931541e-03,1.575298e-10,1.524889e-08,1.797105e-18,1.576937e-36,3.430047e-07,0.006355,0.401520,...,1.752433e-09,5.215746e-01,1.000000e+00,3.629910e-02,5.633998e-24,0.956696,3.372340e-04,2.771238e-01,7.040449e-01,4.007933e-23
ZNF69 (ENSG00000198429)_x,1.297894e-02,7.377750e-01,2.808284e-04,1.118491e-03,8.966391e-01,1.333706e-53,5.624529e-12,0.000000e+00,0.586056,0.000250,...,9.464260e-26,1.176938e-03,3.079091e-07,2.275029e-01,1.000000e+00,0.089834,2.600252e-02,3.491348e-07,7.650499e-02,4.039149e-02
ZNF804A (ENSG00000170396)_x,2.623742e-01,1.935919e-01,5.460522e-02,1.513745e-03,7.688946e-06,2.189568e-05,2.304341e-14,1.096228e-05,0.009784,0.477115,...,3.075805e-01,8.555928e-01,3.969535e-01,4.786208e-02,1.044876e-07,1.000000,1.118015e-02,4.091875e-01,9.928461e-01,7.855822e-11


In [11]:
out_traj_p.to_csv('../../data/processed/out_traj_p.csv')
out_traj_c.to_csv('../../data/processed/out_traj_c.csv')